In [ ]:
!pip install -q segmentation-models-pytorch


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q3-stage3-2026")

print("Path to dataset files:", path)

In [ ]:
def remap_mask(mask):
    # Remaps a mask's pixel values to a consecutive range starting at 0
    mask = mask.long()
    unique_values = torch.unique(mask)
    remapped_mask = torch.zeros_like(mask)

    for new_val, old_val in enumerate(sorted(unique_values.tolist())):
        remapped_mask[mask == old_val] = new_val

    return remapped_mask



In [ ]:
# TO DO

import os
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from PIL import Image
import torchvision.transforms as T
import matplotlib.pyplot as plt

class UnderWaterDataset(Dataset):
    def __confirm_pairs(self):
        assert len(self.img_names) == len(self.mask_names), "Images/Masks count mismatch!"

    def __init__(self, img_dir, mask_dir, size=(256, 256)):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.size = size

        self.img_names = sorted(os.listdir(img_dir))
        self.mask_names = sorted(os.listdir(mask_dir))
        self.__confirm_pairs()

        self.img_tf = T.Compose([
            T.Resize(size),
            T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225]),
        ])

        self.mask_tf = T.Compose([
            T.Resize(size, interpolation=Image.NEAREST),
            T.PILToTensor()
        ])

    def __len__(self):
        return len(self.img_names)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.img_names[idx])
        mask_path = os.path.join(self.mask_dir, self.mask_names[idx])

        img = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path)

        img = self.img_tf(img)
        mask = self.mask_tf(mask).squeeze(0).long()

        mask = remap_mask(mask)

        return img, mask



data_root = os.path.join(path, "dataset")
img_dir = os.path.join(data_root, "images")
mask_dir = os.path.join(data_root, "masks")

full_dataset = UnderWaterDataset(img_dir, mask_dir, size=(256, 256))

# Split 80/20
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, num_workers=2, pin_memory=True)

img, mask = full_dataset[0]
plt.figure(figsize=(10,4))
plt.subplot(1,2,1); plt.imshow(img.permute(1,2,0)); plt.title("Image"); plt.axis("off")
plt.subplot(1,2,2); plt.imshow(mask); plt.title("Mask (remapped)"); plt.axis("off")
plt.show()


In [ ]:
# TO DO
import segmentation_models_pytorch as smp

num_classes = 8

model = smp.Unet(
    encoder_name="efficientnet-b1",
    encoder_weights="imagenet",
    in_channels=3,
    classes=num_classes
)


In [ ]:
# TO DO
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0

    for imgs, masks in loader:
        imgs = imgs.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        optimizer.zero_grad()
        logits = model(imgs)                 # (B, C, H, W)
        loss = criterion(logits, masks)      # masks: (B, H, W) long
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0

    with torch.no_grad():
        for imgs, masks in loader:
            imgs = imgs.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)

            logits = model(imgs)
            loss = criterion(logits, masks)
            total_loss += loss.item()

    return total_loss / len(loader)


In [ ]:
# TO DO
import torch
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

EPOCHS = 10
train_losses, val_losses = [], []

for epoch in range(EPOCHS):
    tr_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
    va_loss = validate(model, val_loader, criterion, device)

    train_losses.append(tr_loss)
    val_losses.append(va_loss)

    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {tr_loss:.4f} | Val Loss: {va_loss:.4f}")

plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Val Loss")
plt.legend()
plt.title("Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.show()


In [ ]:
# TO DO
model.eval()
imgs, masks = next(iter(val_loader))
imgs = imgs.to(device)

with torch.no_grad():
    preds = model(imgs).argmax(dim=1)  # (B,H,W)

imgs = imgs.cpu()
masks = masks.cpu()
preds = preds.cpu()

plt.figure(figsize=(12, 10))
n = min(3, imgs.size(0))
for i in range(n):
    plt.subplot(n, 3, 3*i + 1)
    plt.imshow(imgs[i].permute(1,2,0))
    plt.title("Image"); plt.axis("off")

    plt.subplot(n, 3, 3*i + 2)
    plt.imshow(masks[i])
    plt.title("GT Mask"); plt.axis("off")

    plt.subplot(n, 3, 3*i + 3)
    plt.imshow(preds[i])
    plt.title("Prediction"); plt.axis("off")

plt.tight_layout()
plt.show()
